# bed-pileup
Stack bed file outputs from AmpliconClassifier to make a histogram of ecDNA amplifications across the genome. See **Fig. 2a**.  

For each patient, intervals of the given amplicon class(es) are pooled across all biosamples and merged into a single bed file. Thus, each patient contributes at most once to any genomic position. Which biosample an interval came from is not tracked.

## Requirements
`pyranges` environment  
bed outputs from AmpliconClassifier in `/data/source/AmpliconClassifier/pedpancan_classification_bed_files`

## Results
Per-patient merged .bed files in `beds_by_patient/{run_name}/`  
Output .bedgraph files in `bedgraph/`

In [ ]:
import pandas as pd
import pyranges as pr
import subprocess
import shutil
import re
# Directory setup
import os
from pathlib import Path
# Import local dependencies
import sys
sys.path.append('../../src')
from data_imports import *

import warnings
os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning,ignore::Warning:pyranges.methods.intersection'

In [ ]:
# GLOBALS
BED_DIR = Path("../../data/source/AmpliconClassifier/pedpancan_classification_bed_files") # bed outputs from AmpliconClassifier
PATIENT_BED_DIR = Path('beds_by_patient')
BDG_DIR = Path('bedgraph')
BDG_DIR.mkdir(parents=True, exist_ok=True)
BIOSAMPLES = import_biosamples()
AMPLICON_CLASSES = {
    'ecDNA': ['ecDNA'],
    'intrachromosomal': ['Complex-non-cyclic', 'Linear', 'BFB'],
    'unknown': ['unknown'],
}
BED_NAME = re.compile(r'^(?P<sample_name>.+)_(?P<amplicon_number>amplicon\d+)_(?P<feature_type>.+)_(?P<feature_number>\d+)_intervals\.bed$')

In [ ]:
# Functions

def index_beds(bed_dir=BED_DIR, biosamples=None, verbose=False):
    '''
    Table of AmpliconClassifier bed files with sample, feature type and patient annotations.
    Beds from samples not in biosamples (e.g. cell lines) and empty beds are dropped.
    '''
    if biosamples is None:
        biosamples = import_biosamples()
    rows = []
    empty = []
    for file in sorted(bed_dir.glob('*_intervals.bed')):
        if file.stat().st_size == 0:
            empty.append(file.name)
            continue
        m = BED_NAME.match(file.name)
        if m is None:
            print(f"could not parse bed file name: {file.name}")
            continue
        rows.append({'file': file, **m.groupdict()})
    index = pd.DataFrame(rows)
    in_biosamples = index.sample_name.isin(biosamples.index)
    if verbose:
        print(f"skipping {len(empty)} empty beds.")
        print(f"dropping {(~in_biosamples).sum()} beds from {index[~in_biosamples].sample_name.nunique()} samples not in biosamples:",
              sorted(index[~in_biosamples].sample_name.unique()))
    index = index[in_biosamples]
    return index.merge(biosamples[['patient_id', 'cancer_type']], left_on='sample_name', right_index=True)

def write_patient_beds(run_name, bed_index, verbose=False):
    '''
    Pool and merge intervals across all beds of each patient in bed_index; write one bed per patient
    to PATIENT_BED_DIR/run_name. The output directory is cleared first.
    '''
    dirname = PATIENT_BED_DIR/run_name
    if dirname.exists():
        shutil.rmtree(dirname)
    dirname.mkdir(parents=True)
    for patient_id, files in bed_index.groupby('patient_id').file:
        intervals = pd.concat([pd.read_csv(f, sep='\t', header=None, usecols=[0, 1, 2], names=['Chromosome', 'Start', 'End'])
                               for f in files])
        with warnings.catch_warnings(action="ignore", category=FutureWarning):
            merged = pr.PyRanges(intervals).merge().df.sort_values(['Chromosome', 'Start'])
        merged[['Chromosome', 'Start', 'End']].to_csv(dirname/f'{patient_id}.bed', sep='\t', header=False, index=False)
    if verbose:
        print(f"{run_name}: wrote {bed_index.patient_id.nunique()} patient beds from {len(bed_index)} input beds.")
    return dirname

def run_executable(executable_and_args_list):
    '''
    Inputs: executable and args as a list, eg:
        ['myscript', '-d', 'dparam', '-o', 'oparam']
    '''
    command = [sys.executable] + executable_and_args_list
    try:
        # Run the script and capture output and errors
        result = subprocess.run(command, capture_output=True, text=True)
        # Print the script's output
        print(result.stdout)
        # Print any errors that occurred
        if result.stderr:
            print(f"Errors:\n{result.stderr}", file=sys.stderr)
        # Return the result object, which contains info about the execution
        return result
    except Exception as e:
        print(f"Failed to run script: {e}", file=sys.stderr)
        return None

def run_bed_pileup(bed_dir,outfile):
    # parse inputs
    script_path = Path('..','..','src','bed_pileup.py')
    outfile = Path(outfile)
    command = [script_path, '-d', bed_dir, '-o', BDG_DIR/outfile]
    result = run_executable(command)
    return result

def make_bdg(amplicon_class, cancer_type=None, bed_index=None, verbose=False):
    '''
    amplicon_class: key of AMPLICON_CLASSES.
    cancer_type: optional; restrict to biosamples of this cancer type before merging by patient.
    '''
    if bed_index is None:
        bed_index = index_beds(biosamples=BIOSAMPLES, verbose=verbose)
    run_name = f"{amplicon_class}_{cancer_type or 'all'}"
    subset = bed_index[bed_index.feature_type.isin(AMPLICON_CLASSES[amplicon_class])]
    if cancer_type is not None:
        subset = subset[subset.cancer_type == cancer_type]
    dirname = write_patient_beds(run_name, subset, verbose=verbose)
    run_bed_pileup(dirname, run_name+'.bdg')
    return

In [ ]:
BED_INDEX = index_beds(biosamples=BIOSAMPLES, verbose=True)
for amplicon_class in AMPLICON_CLASSES:
    make_bdg(amplicon_class, bed_index=BED_INDEX, verbose=True)

## Tumor type specific

In [ ]:
for amplicon_class in AMPLICON_CLASSES:
    make_bdg(amplicon_class, cancer_type='EMBT', bed_index=BED_INDEX, verbose=True)